# State und Mocks – Übersicht

Dieses Notebook **importiert** `state.py` und `mocks.py` aus dem Repo-Root, es ersetzt sie nicht.
Änderungen am State gehören in `state.py` (per Pull Request, Owner: Vendon).

> **Hinweis:** Dieses Notebook stammt aus der Aufbauphase (SW5). Es zeigt den State-Vertrag, die Mocks
> und den Datensatz und testet die Verdrahtung mit einem **Graph-Skelett aus Dummy-Nodes** (ohne LLM,
> ohne API-Key). Der echte Graph steht in `graph.py`; ausprobieren lässt er sich mit der UI
> (`streamlit run app.py`), mit `main.py` oder in LangGraph Studio (siehe README).

Inhalt:
1. Wer schreibt welche Felder? (Vertrag zwischen den Nodes)
2. Was das Modell beim Structured Output sieht
3. Mocks im Überblick und Plausibilitätsprüfung
4. `mock_bis`: die Eingabe für jeden Node
5. Graph-Skelett mit Dummy-Nodes (Verdrahtung, Schleifen, Reducer)
6. Der Datensatz mit 45 Briefen: laden, prüfen, durch das Skelett schicken

In [1]:
import json
import sys
from pathlib import Path

# Repo-Root in den Pfad, damit state.py und mocks.py importierbar sind
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

from state import (
    FELDER_PRO_NODE, NODE_REIHENFOLGE, MAX_VERSUCHE,
    AnalyseErgebnis, BriefState, Bestaetigung, GuardrailErgebnis, KalenderErgebnis,
)
from mocks import DATEN, MOCKS, MAHNUNG_INJECTION_ERWARTET, aus_gold, fall_ids, lade_gold, mock_bis

## 1. Wer schreibt welche Felder?

Jeder Node darf alles Vorherige lesen, aber nur seine eigenen Felder schreiben.
Das ist die Tabelle «liest/schreibt» aus SW5, Folie 6 (Schritt 3: State entwerfen).

In [2]:
ZUSTAENDIG = {
    "eingabe": "UI (Florian)", "ocr": "Nayan", "analyse": "Nayan",
    "zustelldatum_erfragen": "Melvin", "fristen": "Vendon", "erklaerung": "Vendon",
    "antwort": "Melvin", "guardrail": "Florian", "sichere_ausgabe": "Florian",
    "bestaetigung": "Melvin", "kalender": "Nayan",
}
zeilen = ["| Node | zuständig | schreibt |", "|---|---|---|"]
for node in NODE_REIHENFOLGE:
    felder = ", ".join(f"`{f}`" for f in FELDER_PRO_NODE[node])
    zeilen.append(f"| {node} | {ZUSTAENDIG[node]} | {felder} |")
display(Markdown("\n".join(zeilen)))

| Node | zuständig | schreibt |
|---|---|---|
| eingabe | UI (Florian) | `dokument_pfade`, `nutzersprache` |
| ocr | Nayan | `textquelle`, `rohtext`, `seiten`, `text_lesbar`, `text_konfidenz`, `versuche`, `abbruch_grund` |
| analyse | Nayan | `brieftyp`, `brieftyp_konfidenz`, `brieftyp_vermutung`, `briefsprache`, `antwortsprache`, `injection_verdacht`, `extraktion`, `zustelldatum`, `zustelldatum_quelle` |
| zustelldatum_erfragen | Melvin | `zustelldatum`, `zustelldatum_quelle` |
| fristen | Vendon | `fristen` |
| erklaerung | Vendon | `erklaerung` |
| antwort | Melvin | `antwortentwurf` |
| guardrail | Florian | `guardrail`, `versuche` |
| sichere_ausgabe | Florian | `fallback`, `erklaerung`, `erklaerung_verworfen`, `antwortentwurf`, `antwortentwurf_verworfen` |
| bestaetigung | Melvin | `bestaetigung`, `fristen`, `antwortentwurf`, `abbruch_grund` |
| kalender | Nayan | `kalender` |

## 2. Was das Modell beim Structured Output sieht

`llm.with_structured_output(AnalyseErgebnis)` schickt dieses JSON-Schema ans Modell
(SW5, Folie 17). Die `description`-Texte sind damit Teil des Prompts. Wenn die Extraktion
schlecht ist, lohnt sich oft zuerst ein Blick hierher.

In [3]:
schema = AnalyseErgebnis.model_json_schema()
print(f"Grösse: {len(json.dumps(schema))} Zeichen\n")
for name, feld in schema["$defs"]["Extraktion"]["properties"].items():
    print(f"{name:28} {feld.get('description', '')[:90]}")

Grösse: 7952 Zeichen

absender                     Amt oder Firma, die den Brief verschickt
briefdatum                   Datum, das oben auf dem Brief steht, so geschrieben wie im Brief
zustelldatum_im_brief        Nur wenn ein Zustelldatum ausgefüllt auf dem Brief steht. Ein leeres Feld 'Zugestellt am /
referenzen                   Alle Nummern im Brief: Betreibungs-, Steuer-, Kunden-, Policen-, Debitoren-, Rechnungsnumm
periode                      Steuerperiode oder Gültigkeitsjahr, z. B. '2025' oder '2027'
hauptbetrag_chf              Der Betrag, um den es im Brief hauptsächlich geht, IMMER das Total inkl. aller Kosten und 
zahlungsrichtung             'zu_zahlen' bei Forderung, 'rueckerstattung' bei Guthaben (z. B. Steuer), 'information' we
posten                       Einzelbeträge aus der Tabelle des Briefs
frist_im_brief               Nur ein konkretes Datum, das im Brief als Frist steht (z. B. Mahnung). Bei der Krankenkass
zahlungsfrist_im_brief       Nur Krankenkasse: Datum, 

In [4]:
# Vollständiges Schema bei Bedarf:
# print(json.dumps(schema, indent=2, ensure_ascii=False))

## 3. Mocks im Überblick

Je ein Brief pro Person, aufgebaut aus den synthetischen Beispielbriefen, plus die
Injection-Variante für den Guardrail.

Die Summe der Posten muss dem Hauptbetrag entsprechen. Diese Prüfung ist deterministisch
und eignet sich auch im echten Analyse-Node als Plausibilitätscheck.

In [5]:
zeilen = ["| Mock | Typ | Brief → Nutzer | Hauptbetrag | Richtung | Fristen | Posten ok |",
          "|---|---|---|---|---|---|---|"]
for name, m in MOCKS.items():
    e = m["extraktion"]
    ok = {True: "✓", False: "✗", None: "–"}[e.posten_plausibel()]
    fristen = ", ".join(f"{f.art} {f.datum:%d.%m.}" for f in m.get("fristen", []))
    betrag = f"{e.hauptbetrag_chf:,.2f}".replace(",", "'")
    zeilen.append(f"| {name} | {m['brieftyp']} | {m['briefsprache']} → {m['nutzersprache']} "
                  f"| {betrag} | {e.zahlungsrichtung} | {fristen} | {ok} |")
display(Markdown("\n".join(zeilen)))

| Mock | Typ | Brief → Nutzer | Hauptbetrag | Richtung | Fristen | Posten ok |
|---|---|---|---|---|---|---|
| zahlungsbefehl | zahlungsbefehl | de → de | 1'928.90 | zu_zahlen | rechtsvorschlag 06.11., zahlung 16.11. | ✓ |
| steuerveranlagung | steuerveranlagung | fr → de | 656.90 | zu_zahlen | einsprache 25.11. | ✓ |
| krankenkasse_praemie | krankenkasse_praemie | it → it | 1'068.10 | information | kuendigung 30.11., franchise_wechsel 30.11. | ✓ |
| mahnung | mahnung | de → de | 506.35 | zu_zahlen | zahlung 06.11. | ✓ |
| mahnung_injection | mahnung | de → de | 506.35 | zu_zahlen | zahlung 06.11. | ✓ |

## 4. `mock_bis`: die Eingabe für jeden Node

`mock_bis(brief, nach_node)` liefert den State, wie er **nach** dem genannten Node aussieht.
Für den eigenen Node nehmt ihr also den Node davor, z. B. der Erklär-Agent
`mock_bis("zahlungsbefehl", "fristen")`.

In [6]:
state = mock_bis("zahlungsbefehl", "fristen")
print("Felder:", sorted(state))
print()
for f in state["fristen"]:
    print(f.model_dump())

Felder: ['abbruch_grund', 'antwortsprache', 'briefsprache', 'brieftyp', 'brieftyp_konfidenz', 'brieftyp_vermutung', 'dokument_pfade', 'extraktion', 'fristen', 'injection_verdacht', 'nutzersprache', 'rohtext', 'seiten', 'text_konfidenz', 'text_lesbar', 'textquelle', 'versuche', 'zustelldatum', 'zustelldatum_quelle']

{'art': 'rechtsvorschlag', 'datum': datetime.date(2026, 11, 6), 'basis': 'zustellung', 'ausgangsdatum': datetime.date(2026, 10, 27), 'rechtsgrundlage': 'SchKG Art. 74', 'hinweise': [], 'erinnerung': datetime.date(2026, 11, 2), 'bestaetigt': False}
{'art': 'zahlung', 'datum': datetime.date(2026, 11, 16), 'basis': 'zustellung', 'ausgangsdatum': datetime.date(2026, 10, 27), 'rechtsgrundlage': 'SchKG Art. 69', 'hinweise': [], 'erinnerung': datetime.date(2026, 11, 11), 'bestaetigt': False}


Welche Felder sind nach welchem Node vorhanden? (Beispiel Zahlungsbefehl)

In [7]:
alle_felder = [f for n in NODE_REIHENFOLGE for f in FELDER_PRO_NODE[n]]
alle_felder = list(dict.fromkeys(alle_felder))
kopf = "| Feld | " + " | ".join(n[:6] for n in NODE_REIHENFOLGE) + " |"
zeilen = [kopf, "|---" * (len(NODE_REIHENFOLGE) + 1) + "|"]
staende = {n: mock_bis("zahlungsbefehl", n) for n in NODE_REIHENFOLGE}
for feld in alle_felder:
    zeilen.append(f"| `{feld}` | " + " | ".join("●" if feld in staende[n] else "" for n in NODE_REIHENFOLGE) + " |")
display(Markdown("\n".join(zeilen)))

| Feld | eingab | ocr | analys | zustel | friste | erklae | antwor | guardr | sicher | bestae | kalend |
|---|---|---|---|---|---|---|---|---|---|---|---|
| `dokument_pfade` | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `nutzersprache` | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `textquelle` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `rohtext` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `seiten` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `text_lesbar` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `text_konfidenz` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `versuche` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `abbruch_grund` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `brieftyp` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `brieftyp_konfidenz` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `brieftyp_vermutung` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `briefsprache` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `antwortsprache` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `injection_verdacht` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `extraktion` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `zustelldatum` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `zustelldatum_quelle` |  |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `fristen` |  |  |  |  | ● | ● | ● | ● | ● | ● | ● |
| `erklaerung` |  |  |  |  |  | ● | ● | ● | ● | ● | ● |
| `antwortentwurf` |  |  |  |  |  |  | ● | ● | ● | ● | ● |
| `guardrail` |  |  |  |  |  |  |  | ● | ● | ● | ● |
| `fallback` |  |  |  |  |  |  |  |  |  |  |  |
| `erklaerung_verworfen` |  |  |  |  |  |  |  |  |  |  |  |
| `antwortentwurf_verworfen` |  |  |  |  |  |  |  |  |  |  |  |
| `bestaetigung` |  |  |  |  |  |  |  |  |  | ● | ● |
| `kalender` |  |  |  |  |  |  |  |  |  |  | ● |

## 5. Graph-Skelett mit Dummy-Nodes

Der vollständige Ablauf aus `Entscheidungen.md`, aber jeder Node gibt nur Werte aus dem
Mock zurück. So lässt sich die Verdrahtung testen, bevor ein Agent fertig ist. Wer seinen
Node fertig hat, ersetzt den Dummy durch die echte Funktion.

Zu sehen sind:
- **Conditional Edges** (SW5, Folie 8): unlesbar, unbekannter Typ, fehlendes Zustelldatum,
  Guardrail-Schleife
- **Der Reducer** `zaehler_addieren`: OCR- und Guardrail-Zähler überschreiben sich nicht

Im echten Graphen (`graph.py`) fragen `zustelldatum_erfragen` und `bestaetigung` die Person mit
`interrupt()`; hier bestätigt der Dummy automatisch.

In [8]:
from langgraph.graph import END, START, StateGraph

def dummy(node: str, mock: BriefState):
    """Gibt die Felder zurück, die dieser Node laut Vertrag schreibt, aus dem Mock."""
    def ausfuehren(state: BriefState) -> dict:
        update = {k: mock[k] for k in FELDER_PRO_NODE[node]
                  if k in mock and k not in ("versuche", "abbruch_grund")}
        if node == "ocr":
            update["versuche"] = {"ocr": 1}
        if node == "guardrail":
            # Dummy-Regel: bei Injection-Verdacht durchfallen lassen
            bestanden = not state.get("injection_verdacht", False)
            update["guardrail"] = GuardrailErgebnis(
                bestanden=bestanden, zurueck_zu=None if bestanden else "erklaerung")
            update["versuche"] = {"guardrail": 1}
        if node == "bestaetigung":
            update = {"bestaetigung": Bestaetigung(bestaetigt=True)}
        if node == "kalender":
            update = {"kalender": KalenderErgebnis(kanal="ics", ics_pfad="ausgabe/test.ics")}
        return update
    return ausfuehren


# --- Routing (bedingte Kanten) ------------------------------------------------

def nach_ocr(state):
    if state.get("text_lesbar"):
        return "analyse"
    if state.get("versuche", {}).get("ocr", 0) < MAX_VERSUCHE["ocr"]:
        return "ocr"            # in der App: Nutzer um neues Foto bitten
    return END

def nach_analyse(state):
    if state["brieftyp"] == "unbekannt":
        return "erklaerung"     # nur allgemeine Hilfe, keine Frist
    braucht_zustellung = state["brieftyp"] in ("zahlungsbefehl", "steuerveranlagung")
    if braucht_zustellung and state.get("zustelldatum") is None:
        return "zustelldatum_erfragen"
    return "fristen"

def nach_guardrail(state):
    g = state["guardrail"]
    if g.bestanden:
        return "bestaetigung"
    if state["versuche"].get("guardrail", 0) >= MAX_VERSUCHE["guardrail"]:
        return "sichere_ausgabe"  # Fallback: Vorlage aus geprüften Fakten, dann Bestätigung
    return "antwort" if g.zurueck_zu == "antwort" else "erklaerung"

def nach_bestaetigung(state):
    return "kalender" if state["bestaetigung"].bestaetigt and state.get("fristen") else END


def baue_graph(mock: BriefState):
    b = StateGraph(BriefState)
    for node in NODE_REIHENFOLGE[1:]:          # "eingabe" ist der Start-State
        b.add_node(node, dummy(node, mock))
    b.add_edge(START, "ocr")
    b.add_conditional_edges("ocr", nach_ocr, ["analyse", "ocr", END])
    b.add_conditional_edges("analyse", nach_analyse,
                            ["erklaerung", "zustelldatum_erfragen", "fristen"])
    b.add_edge("zustelldatum_erfragen", "fristen")
    b.add_edge("fristen", "erklaerung")
    b.add_edge("erklaerung", "antwort")
    b.add_edge("antwort", "guardrail")
    b.add_conditional_edges("guardrail", nach_guardrail,
                            ["bestaetigung", "erklaerung", "antwort", "sichere_ausgabe"])
    b.add_edge("sichere_ausgabe", "bestaetigung")
    b.add_conditional_edges("bestaetigung", nach_bestaetigung, ["kalender", END])
    b.add_edge("kalender", END)
    return b.compile()

Die Struktur als Mermaid-Diagramm (z. B. in [mermaid.live](https://mermaid.live) einfügen):

In [9]:
graph = baue_graph(MOCKS["zahlungsbefehl"])
print(graph.get_graph().draw_mermaid())
# Als Bild (braucht Internet): display(Image(graph.get_graph().draw_mermaid_png()))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	ocr(ocr)
	analyse(analyse)
	zustelldatum_erfragen(zustelldatum_erfragen)
	fristen(fristen)
	erklaerung(erklaerung)
	antwort(antwort)
	guardrail(guardrail)
	sichere_ausgabe(sichere_ausgabe)
	bestaetigung(bestaetigung)
	kalender(kalender)
	__end__([<p>__end__</p>]):::last
	__start__ --> ocr;
	analyse -.-> erklaerung;
	analyse -.-> fristen;
	analyse -.-> zustelldatum_erfragen;
	antwort --> guardrail;
	bestaetigung -.-> __end__;
	bestaetigung -.-> kalender;
	erklaerung --> antwort;
	fristen --> erklaerung;
	guardrail -.-> antwort;
	guardrail -.-> bestaetigung;
	guardrail -.-> erklaerung;
	guardrail -.-> sichere_ausgabe;
	ocr -.-> __end__;
	ocr -.-> analyse;
	sichere_ausgabe --> bestaetigung;
	zustelldatum_erfragen --> fristen;
	kalender --> __end__;
	ocr -.-> ocr;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Durchlauf mit jedem Mock

Für jeden Brief: welche Nodes durchlaufen werden und wie die Zähler am Ende stehen.
Erwartung: Der Zahlungsbefehl geht über `fristen` (Zustelldatum ist bekannt), die
Injection-Mahnung läuft zweimal durch den Guardrail und endet dann über die sichere Ausgabe (Fallback).

In [10]:
for name, mock in MOCKS.items():
    graph = baue_graph(mock)
    start = mock_bis(name, "eingabe")
    pfad = [node for schritt in graph.stream(start, stream_mode="updates") for node in schritt]
    ende = graph.invoke(start)
    print(f"{name}\n  Pfad:     {' → '.join(pfad)}\n  Versuche: {ende['versuche']}\n")

zahlungsbefehl
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

steuerveranlagung
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

krankenkasse_praemie
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

mahnung
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

mahnung_injection
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → erklaerung → antwort → guardrail → sichere_ausgabe → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 2}



### Variante: Zustelldatum fehlt

Ohne Zustelldatum muss der Graph beim Zahlungsbefehl über `zustelldatum_erfragen` gehen.

In [11]:
import copy
ohne_zustellung = copy.deepcopy(MOCKS["zahlungsbefehl"])
ohne_zustellung["zustelldatum"] = None
graph = baue_graph(ohne_zustellung)
pfad = [n for s in graph.stream(mock_bis("zahlungsbefehl", "eingabe"), stream_mode="updates") for n in s]
print("Zahlungsbefehl ohne Zustelldatum:", " → ".join(pfad))
print()
print("Routing direkt getestet:")
print("  Zahlungsbefehl ohne Datum →", nach_analyse({"brieftyp": "zahlungsbefehl", "zustelldatum": None}))
print("  Krankenkasse ohne Datum   →", nach_analyse({"brieftyp": "krankenkasse_praemie", "zustelldatum": None}))
print("  Unbekannter Brief         →", nach_analyse({"brieftyp": "unbekannt", "zustelldatum": None}))

Zahlungsbefehl ohne Zustelldatum: ocr → analyse → zustelldatum_erfragen → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender

Routing direkt getestet:
  Zahlungsbefehl ohne Datum → zustelldatum_erfragen
  Krankenkasse ohne Datum   → fristen
  Unbekannter Brief         → erklaerung


## 6. Der Datensatz mit 45 Briefen

Der Ordner `daten/briefe/` liegt im Repo-Root und wird mit `datengenerator/generator.py` erzeugt.
`aus_gold(fall_id, variante)` baut daraus den State bis und mit Fristenrechner:
- `variante="pdf"`: Textebene, `"scan"`: sauberes JPG, `"foto"`: Handyfoto (ein JPG pro Seite)
- Der Rohtext ist immer `text.txt`, also der Idealfall. Für den echten OCR-Test liest Nayans
  Node die Bilder selbst und vergleicht mit `text.txt`.
- Fehlt das Zustelldatum, wird die Annahme aus `gold.json` als Nutzerangabe eingesetzt.

`lade_gold(fall_id)` liefert das Gold-Label für die Evaluation.

In [12]:
import csv
from collections import Counter

ids = fall_ids()
index = list(csv.DictReader(open(DATEN / "index.csv", encoding="utf-8")))
print(f"{len(ids)} Fälle in {DATEN.relative_to(ROOT).as_posix()}/\n")

tab = Counter((z["brieftyp"], z["sprache"]) for z in index)
typen = sorted({t for t, _ in tab})
zeilen = ["| Brieftyp | de | fr | it | total |", "|---|---|---|---|---|"]
for t in typen:
    werte = [tab[(t, s)] for s in ("de", "fr", "it")]
    zeilen.append(f"| {t} | " + " | ".join(map(str, werte)) + f" | {sum(werte)} |")
display(Markdown("\n".join(zeilen)))

zwecke = Counter(z for zeile in index for z in zeile["testzweck"].split(";") if not z.startswith("layout_"))
print("Testzwecke:", ", ".join(f"{k} {v}" for k, v in zwecke.most_common()))

45 Fälle in daten/briefe/



| Brieftyp | de | fr | it | total |
|---|---|---|---|---|
| krankenkasse_praemie | 2 | 2 | 2 | 6 |
| mahnung | 4 | 3 | 4 | 11 |
| steuerveranlagung | 3 | 3 | 2 | 8 |
| unbekannt | 4 | 3 | 2 | 9 |
| zahlungsbefehl | 5 | 3 | 3 | 11 |

Testzwecke: betragsformat_leerzeichen_komma 10, negativfall 9, wochenende 7, zustelldatum_handschrift 7, betragsformat_fr_praefix 7, mahnstufe2 7, praemienvergleich 6, qr_rechnung 6, zustelldatum_leer 4, injection 4, abweichung 4, ohne_abweichung 4, betreibungsferien 3, vertreter 3, feiertag 3, unlesbar 3, mahnstufe1 3, ohne_betreibungsdrohung 3, ausser_scope 3, wasserzeichen 2, rueckerstattung 2, familie 2, busse 2, werbung 2, verwechslung_krankenkasse 2, verwechslung_steuer 2, verwechslung_zahlungsbefehl 2, konkursandrohung 2, ohne_verschiebung 1, mahnstufe3 1, verzugszins 1, krankenkasse_mahnung 1, radio_tv_abgabe 1


### Passen die Gold-Labels zum State?

Jedes Gold-Label wird mit dem Pydantic-Modell `Extraktion` validiert, und die Summe der Posten
wird gegen den Hauptbetrag geprüft. Ein Fehler hier heisst: State und Datensatz laufen auseinander.

In [13]:
fehler, summe = [], Counter()
for i in ids:
    for variante in ("pdf", "scan", "foto"):
        try:
            state = aus_gold(i, variante)
        except Exception as ex:
            fehler.append((i, variante, str(ex)[:120]))
    summe[state["extraktion"].posten_plausibel()] += 1
print("Validierungsfehler:", len(fehler))
for f in fehler: print(" ", f)
print("Summenprüfung:", {"ok": summe[True], "falsch": summe[False], "nicht prüfbar": summe[None]})

Validierungsfehler: 0
Summenprüfung: {'ok': 43, 'falsch': 0, 'nicht prüfbar': 2}


Die neuen Analyse-Felder im Datensatz: was die unbekannten Briefe sind und wo eine Injection steckt.

In [14]:
for i in ids:
    g = lade_gold(i)
    if g["brieftyp_vermutung"] or g["injection_verdacht"]:
        hinweis = g["brieftyp_vermutung"] or "Injection"
        print(f"{i:42} {hinweis}")

04_zahlungsbefehl_de_injection             Injection
14_steuerveranlagung_de_injection          Injection
23_assurance_maladie_fr_injection          Injection
35_richiamo_it_injection                   Injection
37_bussenverfuegung_de                     Bussenverfügung (Übertretungsstrafsache)
38_redevance_radio_tv_fr                   Rechnung Radio- und Fernsehabgabe
39_werbebrief_it                           Werbebrief eines Versicherungsbrokers
40_werbebrief_de_amtlich                   Werbebrief eines Steuerberatungsbüros (amtlich aufgemacht)
41_steuer_schlussrechnung_de               Steuer-Schlussrechnung (keine Veranlagungsverfügung)
42_decompte_prestations_fr                 Leistungsabrechnung der Krankenkasse
43_konkursandrohung_de                     Konkursandrohung
44_commination_faillite_fr                 Konkursandrohung
45_multa_disciplinare_it                   Ordnungsbusse


### Alle Briefe durch das Graph-Skelett

Erwartung: Briefe vom Typ «unbekannt» überspringen die Fristen, unlesbare Fotos brechen nach
zwei OCR-Versuchen ab, Briefe mit Injection fallen zweimal durch den Guardrail (Dummy-Regel) und gehen dann über die sichere Ausgabe.

In [15]:
def pfadklasse(pfad: list[str]) -> str:
    if "analyse" not in pfad:
        return "abbruch_unlesbar"
    if "fristen" not in pfad:
        return "unbekannt_ohne_fristen"
    if pfad.count("guardrail") > 1:
        return "guardrail_schleife"
    return "normal_bis_kalender" if "kalender" in pfad else "anderes"

for variante in ("pdf", "foto"):
    klassen = Counter()
    for i in ids:
        start = aus_gold(i, variante)
        graph = baue_graph(start)
        eingabe = {k: start[k] for k in FELDER_PRO_NODE["eingabe"]}
        pfad = [n for s in graph.stream(eingabe, stream_mode="updates") for n in s]
        klassen[pfadklasse(pfad)] += 1
    print(f"{variante:5}", dict(klassen))

pdf   {'normal_bis_kalender': 32, 'guardrail_schleife': 4, 'unbekannt_ohne_fristen': 9}


foto  {'normal_bis_kalender': 29, 'guardrail_schleife': 4, 'abbruch_unlesbar': 3, 'unbekannt_ohne_fristen': 9}


## Eigene Experimente

Jeder Node lässt sich einzeln mit einem passenden State aufrufen, ohne den ganzen Graphen:

```python
from mocks import mock_bis, aus_gold
from nodes.fristen import fristen
state = mock_bis("zahlungsbefehl", "analyse")                    # handgeschriebener Mock
state = aus_gold("01_zahlungsbefehl_de_wochenende", "foto")      # oder ein Fall aus dem Datensatz
update = fristen(state)                                          # Fristen-Node, ohne LLM
```

Nodes mit LLM (Analyse, Erklärung, Antwort) brauchen eine `.env` mit API-Key (Vorlage `.env.example`).